###### AutoEncoder — 3A to 3F

Dense, convolutional and LSTM autoencoders are selected automatically from TASK. The data loader follows each final problem statement.

In [47]:
import numpy as np, pandas as pd, tensorflow as tf, os, glob
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
# 3A CreditCard | 3B ECG | 3C SmartGrid | 3D NASA | 3E ECG | 3F SignMNIST


In [55]:
#3A
# data = pd.read_csv('creditcard.csv')
# X, y = data.drop('Class', axis=1), data['Class']

# X = StandardScaler().fit_transform(X)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )
# X_train = X_train[y_train == 0]
# mode = "dense"

#3B
# data = pd.read_csv("ecg.csv")

# y = data.iloc[:,-1]
# X = data.iloc[:,:-1]

# X = StandardScaler().fit_transform(X)

# X_train, X_test, y_train, y_test = train_test_split(
#     X,y,test_size=0.2,random_state=42)

# X_train = X_train[y_train==0]
# mode = "dense"

#3C
# data = pd.read_csv("energy consumption.csv").select_dtypes(include=np.number).ffill().bfill()
# X = StandardScaler().fit_transform(data)

# X = np.array([X[i:i+20] for i in range(len(X)-19)])
# X_train, X_test = train_test_split(X,test_size=0.2,shuffle=False)
# mode = "lstm"

#3D
# data = pd.read_csv("NASA TurboFan Engine.csv")

# y = data["Fault_Label"]
# X = StandardScaler().fit_transform(data.drop(columns=["unit","cycle","RUL","Fault_Label"]).ffill().bfill())

# X = np.array([X[i:i+20] for i in range(len(X)-19)])
# y = np.array([max(y[i:i+20]) for i in range(len(y)-19)])

# X_train, X_test, y_train, y_test = train_test_split(
#     X,y,test_size=0.2,shuffle=False
# )

# X_train = X_train[y_train==0]
# mode = "lstm"

#3E
# tr = pd.read_csv("sign_mnist_train.csv")
# te = pd.read_csv("sign_mnist_test.csv")

# X_train = tr.iloc[:,1:].values.astype("float32")/255
# X_test = te.iloc[:,1:].values.astype("float32")/255

# y_train = tr.iloc[:,0].values
# y_test = (te.iloc[:,0].values==24).astype(int)

# X_train = X_train[y_train!=24]
# X_train = X_train.reshape(-1,28,28,1)
# X_test = X_test.reshape(-1,28,28,1)
# mode = "cnn"

In [56]:
inp = layers.Input(shape=X_train.shape[1:])
if mode == "dense":
    x = layers.Flatten()(inp)
    x = layers.Dense(32,activation="relu")(x)
    x = layers.Dense(8,activation="relu")(x)
    x = layers.Dense(32,activation="relu")(x)
    x = layers.Dense(int(np.prod(X_train.shape[1:])))(x)
    out = layers.Reshape(X_train.shape[1:])(x)
elif mode == "cnn":
    x = layers.Conv2D(16,3,2,padding="same",activation="relu")(inp)
    x = layers.Conv2D(8,3,2,padding="same",activation="relu")(x)
    x = layers.Conv2DTranspose(16,3,2,padding="same",activation="relu")(x)
    out = layers.Conv2DTranspose(3,3,2,padding="same",activation="sigmoid")(x)
else:
    x = layers.LSTM(32,return_sequences=True)(inp)
    x = layers.LSTM(8)(x)
    x = layers.RepeatVector(X_train.shape[1])(x)
    x = layers.LSTM(32,return_sequences=True)(x)
    out = layers.LSTM(X_train.shape[-1],return_sequences=True)(x)

model = Model(inp,out)
model.compile(optimizer="adam",loss="mse")
model.fit(X_train,X_train,epochs=10,batch_size=64,validation_split=0.2)


Epoch 1/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0195 - val_loss: 0.0107
Epoch 2/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0073 - val_loss: 0.0040
Epoch 3/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0030 - val_loss: 0.0023
Epoch 4/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0021 - val_loss: 0.0019
Epoch 5/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 6/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 7/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 8/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 9/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0014 - val_loss: 0.0013
Epoch 10/10
330/330 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0013 - val_loss: 0.0013


In [57]:
# Reconstruction error and anomaly decision
def error(a):
    r = model.predict(a,verbose=0)
    return np.mean((a-r)**2,axis=tuple(range(1,a.ndim)))

threshold = np.percentile(error(X_train),95)
test_error = error(X_test)
pred = (test_error > threshold).astype(int)
print("Threshold:",threshold)
print("Anomalies detected:",sum(pred))
print("Predictions:",pred[:30])

if "y_test" in locals() and len(y_test)==len(pred):
    print("Confusion Matrix:\n",confusion_matrix(y_test,pred))
    print(classification_report(y_test,pred,zero_division=0))


Threshold: 0.0026302803
Anomalies detected: 273
Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Confusion Matrix:
 [[6582  258]
 [ 317   15]]
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      6840
           1       0.05      0.05      0.05       332

    accuracy                           0.92      7172
   macro avg       0.50      0.50      0.50      7172
weighted avg       0.91      0.92      0.92      7172



In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf, os, glob
from tensorflow.keras import Model, layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [4]:
data = pd.read_csv('creditcard.csv')
X,y = data.drop("Class", axis =1), data["Class"]
X = StandardScaler().fit_transform(X)

X_train, X_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[y_train==0]
mode = "dense"

In [8]:
inp = layers.Input(shape=(X_train.shape[1],))
if mode == "dense" :
    x = layers.Flatten()(inp)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(8, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(int(np.prod(X_train.shape[1:])))(x)
    out = layers.Reshape(X_train.shape[1:])(x)

In [9]:
model = Model(inp, out)
model.summary()
model.compile( optimizer="sgd", loss="mse", metrics=["mae"] )
history = model.fit(X_train, X_train, batch_size=64, epochs=10, validation_split=0.1)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 30)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 30)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │             992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8)                   │             264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 30)                  │             990 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 30)                  │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,534 (9.90 KB)

 Trainable params: 2,534 (9.90 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 0.9178 - mae: 0.6407 - val_loss: 0.8309 - val_mae: 0.6194
Epoch 2/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.7677 - mae: 0.5978 - val_loss: 0.7006 - val_mae: 0.5814
Epoch 3/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.6808 - mae: 0.5722 - val_loss: 0.6393 - val_mae: 0.5611
Epoch 4/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.6188 - mae: 0.5515 - val_loss: 0.5888 - val_mae: 0.5422
Epoch 5/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.5745 - mae: 0.5345 - val_loss: 0.5506 - val_mae: 0.5269
Epoch 6/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.5374 - mae: 0.5195 - val_loss: 0.5149 - val_mae: 0.5120
Epoch 7/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.5034 - mae: 0.5050 - val_loss: 0.4835 - val_mae: 0.4981
Epoch 8/10
3199/3199 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 0.4744 - mae: 0.4919 - val_loss: 0.4583 - val_mae: 0.4860
Epoch 9/10
3199/3199 ━━━━━━━━━━━━━━━━━━━

In [16]:
def error(a):
    r = model.predict(a, verbose=0)
    return np.mean((a-r)**2,axis=tuple(range(1, a.ndim)))
    
threshold = np.percentile(error(X_train),95)
test_error = error(X_text)
pred = (test_error > threshold).astype("int")

print("Threshold:",threshold)
print("Anomalies detected:",sum(pred))
print("Predictions:",pred[:30])

Threshold: 0.9543959245839577
Anomalies detected: 2931
Predictions: [1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [18]:
print("Confusion Matrix:\n",confusion_matrix(y_test,pred))
print(classification_report(y_test,pred,zero_division=0))

Confusion Matrix:
 [[54017  2847]
 [   14    84]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     56864
           1       0.03      0.86      0.06        98

    accuracy                           0.95     56962
   macro avg       0.51      0.90      0.51     56962
weighted avg       1.00      0.95      0.97     56962



In [20]:
import pandas as pd
import numpy as np
import tensorflow as ts, os, glob
from tensorflow.keras import Model, layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [33]:
data = pd.read_csv("energy consumption.csv").select_dtypes(include=np.number).ffill().bfill()
X = StandardScaler().fit_transform(data)

X = np.array([X[i:i+20] for i in range(len(X)-19)])
X_train, X_test = train_test_split(X, test_size=0.2, shuffle=False,random_state=42)

In [35]:
inp = layers.Input(shape=X_train.shape[1:])
x = layers.LSTM(32,return_sequences=True)(inp)
x = layers.LSTM(8)(x)
x = layers.RepeatVector(X_train.shape[1])(x)
x = layers.LSTM(32,return_sequences=True)(x)
out = layers.LSTM(X_train.shape[-1],return_sequences=True)(x)

In [38]:
model = Model(inp,out)
model.summary()
model.compile(optimizer="adam",loss="mse")
model.fit(X_train,X_train,epochs=10,batch_size=64,validation_split=0.2)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)           │ (None, 20, 6)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 20, 32)              │           4,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 8)                   │           1,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector (RepeatVector)         │ (None, 20, 8)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 20, 32)              │           5,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 20, 6)               │             936 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 12,488 (48.78 KB)

 Trainable params: 12,488 (48.78 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - loss: 0.9405 - val_loss: 0.9800
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.9045 - val_loss: 0.9569
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8826 - val_loss: 0.9330
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8596 - val_loss: 0.9134
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8453 - val_loss: 0.9033
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8369 - val_loss: 0.8958
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8295 - val_loss: 0.8888
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.8226 - val_loss: 0.8822
Epoch 9/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.8165 - val_loss: 0.8762
Epoch 10/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.8119 - val_loss: 0.8722


In [40]:
def error(a):
    r = model.predict(a, verbose=0)
    return np.mean((a-r)**2, axis=tuple(range(1, a.ndim)))
threshold = np.percentile(error(X_train),95)
test_error = error(X_test)
pred = (test_error > threshold).astype("int")

print("Threshold:", threshold)
print("Anomalies Detected:", sum(pred))
print("Predictions :", pred[:30])

Threshold: 1.2787706005287651
Anomalies Detected: 100
Predictions : [1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
from sklearn.metrics import classification_report
print("Confusion Matrix:\n", confusion_matrix(y_test,pred))
print("Accuracy Score:\n", classification_report(y_test,pred,zero_division=0))